# YOLO Fruit Detection Training

Train YOLO v8 model for multi-class fruit detection (Mango, Grapefruit, Guava, Orange)

In [ ]:
# Install Required Packages (Run this first in Google Colab)
!pip install ultralytics
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

print("Packages installed successfully!")

In [ ]:
# Mount Google Drive (For Google Colab)
try:
    from google.colab import drive
    import os
    
    # Mount Google Drive
    drive.mount('/content/drive')
    
    # Change to your dataset directory (adjust path as needed)
    # The dataset is directly in Google Drive MyDrive as multi_fruit_detection
    dataset_path = '/content/drive/MyDrive/multi_fruit_detection'
    
    if os.path.exists(dataset_path):
        print(f"✅ Found dataset directory: {dataset_path}")
        print("Dataset contents:")
        print([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])
    else:
        print(f"❌ Dataset directory not found: {dataset_path}")
        print("Please upload your multi_fruit_detection folder to Google Drive MyDrive")
        print("Available directories in MyDrive:")
        mydrive_path = '/content/drive/MyDrive'
        if os.path.exists(mydrive_path):
            print([d for d in os.listdir(mydrive_path) if os.path.isdir(os.path.join(mydrive_path, d))])
    
    print("Google Drive mounted successfully!")
    
except ImportError:
    print("Not running in Google Colab, skipping drive mount...")
    print("Using local dataset path...")

In [ ]:
# Import Required Libraries
import os
import torch
from ultralytics import YOLO
import yaml
from pathlib import Path
import matplotlib.pyplot as plt
import shutil

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
# Load Dataset Configuration
# Check if we're in Google Colab or local environment
try:
    from google.colab import drive
    # We're in Colab - use Google Drive path
    dataset_path = Path('/content/drive/MyDrive/multi_fruit_detection')
except ImportError:
    # We're local - use local path  
    dataset_path = Path('data/unified/multi_fruit_detection')

config_path = dataset_path / 'data.yaml'

if config_path.exists():
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)

    print("📋 Dataset Configuration Loaded:")
    print(f"Classes: {config['nc']}")
    print(f"Names: {config['names']}")
    print(f"Train path: {config['train']}")
    print(f"Val path: {config['val']}")
    print(f"Full dataset path: {config['path']}")
else:
    print("❌ data.yaml not found!")
    print(f"Looking for: {config_path}")
    print("Please run the previous cells to verify and create the dataset configuration.")

In [ ]:
# Verify Dataset Structure and Create Colab-Compatible data.yaml
import os
from pathlib import Path

# Check if we're in Google Colab or local environment
try:
    from google.colab import drive
    # We're in Colab - use Google Drive path
    dataset_path = Path('/content/drive/MyDrive/multi_fruit_detection')
    print("🔍 Running in Google Colab")
except ImportError:
    # We're local - use local path
    dataset_path = Path('data/unified/multi_fruit_detection')
    print("🔍 Running locally")

print(f"Dataset path: {dataset_path}")
print(f"Dataset exists: {dataset_path.exists()}")

if dataset_path.exists():
    # Check dataset structure
    train_path = dataset_path / 'train' / 'images'
    val_path = dataset_path / 'val' / 'images'
    test_path = dataset_path / 'test' / 'images'
    
    print(f"\n📁 Dataset Structure:")
    print(f"Train images: {len(list(train_path.glob('*.jpg'))) if train_path.exists() else 0}")
    print(f"Val images: {len(list(val_path.glob('*.jpg'))) if val_path.exists() else 0}")
    print(f"Test images: {len(list(test_path.glob('*.jpg'))) if test_path.exists() else 0}")
    
    # Check label files
    train_labels = dataset_path / 'train' / 'labels'
    val_labels = dataset_path / 'val' / 'labels'
    
    print(f"Train labels: {len(list(train_labels.glob('*.txt'))) if train_labels.exists() else 0}")
    print(f"Val labels: {len(list(val_labels.glob('*.txt'))) if val_labels.exists() else 0}")
    
    # List what's actually in the dataset directory
    print(f"\n📋 Contents of dataset directory:")
    items = list(dataset_path.iterdir())
    for item in sorted(items):
        if item.is_dir():
            print(f"📁 {item.name}/")
        else:
            print(f"📄 {item.name}")
    
else:
    print("❌ Dataset not found!")
    print("Please make sure to upload your multi_fruit_detection folder to Google Drive MyDrive")

In [ ]:
# Create Colab-Compatible data.yaml
# Use the dataset_path from the previous cell
if dataset_path.exists():
    # Update data.yaml for current environment (Colab paths)
    yaml_content = {
        'path': str(dataset_path),
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images',
        'nc': 4,
        'names': ['mango', 'grapefruit', 'guava', 'orange']
    }
    
    # Save updated yaml file
    yaml_file = dataset_path / 'data.yaml'
    with open(yaml_file, 'w') as f:
        yaml.dump(yaml_content, f, default_flow_style=False)
    
    print("✅ Updated data.yaml for current environment")
    print(f"Dataset path in yaml: {yaml_content['path']}")
    
    # Verify the yaml file was created correctly
    if yaml_file.exists():
        with open(yaml_file, 'r') as f:
            config = yaml.safe_load(f)
        
        print("\n📋 Dataset Configuration:")
        print(f"Classes: {config['nc']}")
        print(f"Names: {config['names']}")
        print(f"Train path: {config['train']}")
        print(f"Val path: {config['val']}")
        print(f"Full dataset path: {config['path']}")
        
        # Store the yaml path for training
        global yaml_path
        yaml_path = str(yaml_file)
        print(f"\n🎯 YAML file ready for training: {yaml_path}")
    else:
        print("❌ Failed to create data.yaml file")
    
else:
    print("❌ Cannot create data.yaml - dataset not found!")
    print("Please ensure multi_fruit_detection folder is in your Google Drive MyDrive")

In [ ]:
# Initialize YOLO Model
model = YOLO('yolov8n.pt')  # Load pretrained YOLOv8 nano model

print("YOLO model loaded successfully!")
print(f"Model: {model.model}")

In [ ]:
# Verify Dataset Structure and Create Colab-Compatible data.yaml
import os
from pathlib import Path

# Check if we're in Google Colab or local environment
try:
    from google.colab import drive
    # We're in Colab - use Google Drive path
    dataset_path = Path('/content/drive/MyDrive/multi_fruit_detection')
    print("🔍 Running in Google Colab")
except ImportError:
    # We're local - use local path
    dataset_path = Path('data/unified/multi_fruit_detection')
    print("🔍 Running locally")

print(f"Dataset path: {dataset_path}")
print(f"Dataset exists: {dataset_path.exists()}")

if dataset_path.exists():
    # Check dataset structure and count images
    train_path = dataset_path / 'train' / 'images'
    val_path = dataset_path / 'val' / 'images'
    test_path = dataset_path / 'test' / 'images'
    
    # Count images in each set
    train_images = len(list(train_path.glob('*.jpg'))) if train_path.exists() else 0
    val_images = len(list(val_path.glob('*.jpg'))) if val_path.exists() else 0
    test_images = len(list(test_path.glob('*.jpg'))) if test_path.exists() else 0
    total_images = train_images + val_images + test_images
    
    print(f"\n📁 Dataset Structure & Image Counts:")
    print(f"Train images: {train_images}")
    print(f"Val images: {val_images}")
    print(f"Test images: {test_images}")
    print(f"Total images: {total_images}")
    
    # Calculate and display percentages
    if total_images > 0:
        train_pct = (train_images / total_images) * 100
        val_pct = (val_images / total_images) * 100
        test_pct = (test_images / total_images) * 100
        
        print(f"\n📊 Dataset Split Percentages:")
        print(f"Train: {train_pct:.1f}% ({train_images}/{total_images})")
        print(f"Val: {val_pct:.1f}% ({val_images}/{total_images})")
        print(f"Test: {test_pct:.1f}% ({test_images}/{total_images})")
        
        # Check if split is reasonable (typical is 70/15/15 or 80/10/10)
        if train_pct < 60:
            print("⚠️  Warning: Training set seems small (< 60%)")
        elif train_pct > 90:
            print("⚠️  Warning: Training set seems too large (> 90%)")
        else:
            print("✅ Dataset split looks reasonable")
    
    # Check label files
    train_labels = dataset_path / 'train' / 'labels'
    val_labels = dataset_path / 'val' / 'labels'
    test_labels = dataset_path / 'test' / 'labels'
    
    train_labels_count = len(list(train_labels.glob('*.txt'))) if train_labels.exists() else 0
    val_labels_count = len(list(val_labels.glob('*.txt'))) if val_labels.exists() else 0
    test_labels_count = len(list(test_labels.glob('*.txt'))) if test_labels.exists() else 0
    
    print(f"\n?️  Label Files:")
    print(f"Train labels: {train_labels_count}")
    print(f"Val labels: {val_labels_count}")
    print(f"Test labels: {test_labels_count}")
    
    # Verify images and labels match
    print(f"\n🔍 Image-Label Verification:")
    if train_images == train_labels_count:
        print("✅ Train: Images and labels match")
    else:
        print(f"❌ Train: {train_images} images but {train_labels_count} labels")
        
    if val_images == val_labels_count:
        print("✅ Val: Images and labels match")
    else:
        print(f"❌ Val: {val_images} images but {val_labels_count} labels")
        
    if test_images == test_labels_count:
        print("✅ Test: Images and labels match")
    else:
        print(f"❌ Test: {test_images} images but {test_labels_count} labels")
    
    # Overall dataset health check
    print(f"\n🏥 Dataset Health Check:")
    if total_images > 1000:
        print(f"✅ Good dataset size: {total_images} images")
    elif total_images > 500:
        print(f"⚠️  Moderate dataset size: {total_images} images")
    else:
        print(f"❌ Small dataset size: {total_images} images (might need more data)")
    
    # List what's actually in the dataset directory
    print(f"\n📋 Contents of dataset directory:")
    items = list(dataset_path.iterdir())
    for item in sorted(items):
        if item.is_dir():
            print(f"📁 {item.name}/")
        else:
            print(f"📄 {item.name}")
    
else:
    print("❌ Dataset not found!")
    print("Please make sure to upload your multi_fruit_detection folder to Google Drive MyDrive")

In [ ]:
# Evaluate Model and Load Best Weights
import os

# Check if training results exist
results_dir = 'runs/detect/multi_fruit_detection_v1'
best_weights = f'{results_dir}/weights/best.pt'

if os.path.exists(best_weights):
    print("🔍 Evaluating trained model...")
    
    # Evaluate Model
    metrics = model.val()
    print(f"📊 Model Performance:")
    print(f"mAP@0.5: {metrics.box.map50:.4f}")
    print(f"mAP@0.5-0.95: {metrics.box.map:.4f}")
    
    # Load best model
    best_model = YOLO(best_weights)
    print(f"✅ Best model loaded from: {best_weights}")
    
    # Display training curves if available
    results_img = f'{results_dir}/results.png'
    if os.path.exists(results_img):
        from IPython.display import Image, display
        print("\n📈 Training Results:")
        display(Image(results_img))
    
else:
    print("❌ Training results not found!")
    print("Please run the training cell first or check if training completed successfully.")

In [ ]:
# Test Model with Sample Predictions
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# Test on some sample images from validation set
try:
    # Get some test images
    val_images_path = dataset_path / 'val' / 'images'
    sample_images = list(val_images_path.glob('*.jpg'))[:6]  # Get first 6 images
    
    if sample_images and os.path.exists(best_weights):
        print("🔍 Testing model on sample images...")
        
        # Create a subplot for displaying results
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        
        for idx, img_path in enumerate(sample_images):
            # Run inference
            results = best_model(str(img_path))
            
            # Get annotated image
            annotated_img = results[0].plot()
            
            # Convert BGR to RGB for matplotlib
            annotated_img = annotated_img[:, :, ::-1]
            
            # Display
            axes[idx].imshow(annotated_img)
            axes[idx].set_title(f'Sample {idx+1}')
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        print("✅ Sample predictions displayed!")
        
    else:
        print("❌ No sample images found or model not trained yet")
        print(f"Looking for images in: {val_images_path}")
        
except Exception as e:
    print(f"❌ Error during inference: {e}")
    print("Make sure the model is trained and sample images are available.")

In [ ]:
# Export Model for Inference
# Export the trained model in different formats for deployment

if os.path.exists(best_weights):
    print("📦 Exporting trained model...")
    
    # Export to ONNX format (good for deployment)
    try:
        best_model.export(format='onnx')
        print("✅ Model exported to ONNX format")
    except Exception as e:
        print(f"❌ ONNX export failed: {e}")
    
    # Export to TorchScript (good for production)
    try:
        best_model.export(format='torchscript')
        print("✅ Model exported to TorchScript format")
    except Exception as e:
        print(f"❌ TorchScript export failed: {e}")
    
    print(f"\n📁 Model files available in: {results_dir}/weights/")
    
    # Display model summary
    print(f"\n📋 Model Summary:")
    print(f"Classes: {len(config['names'])}")
    print(f"Class names: {config['names']}")
    print(f"Input size: 640x640")
    print(f"Model architecture: YOLOv8 Nano")
    
else:
    print("❌ No trained model found to export!")
    print("Please complete the training first.")